# Ensemble Methods

Ensemble methods combine multiple models to produce stronger predictions. This notebook demonstrates three techniques:

| Method | Key idea |
|--------|----------|
| **Hard Voting** | Several different classifiers vote; majority wins |
| **Bagging** | Many trees, each trained on a bootstrap sample |
| **Random Forest** | Bagging + random feature subsets at each split |

**Dataset:** Wine — classify wine cultivars from chemical features.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
import sys
sys.path.insert(0, '.')
from ensemble import hard_voting_classifier, bagging_classifier, random_forest_classifier
from decision_tree_classifier import decision_tree_classifier
from knn import KNN
from logistic_regression import LogisticRegression
np.random.seed(42)
print("Imports complete")

## Load & Explore the Data

In [ ]:
data = load_wine()
X, y = data.data, data.target
feature_names = data.feature_names
class_names   = data.target_names

print(f"Samples: {X.shape[0]} | Features: {X.shape[1]} | Classes: {len(class_names)}")
print(f"Class distribution: {dict(zip(class_names, [int((y==i).sum()) for i in range(3)]))}")

plt.figure(figsize=(8,4))
plt.bar(class_names, [(y==i).sum() for i in range(3)],
        color=['#e63946','#457b9d','#2a9d8f'], edgecolor='white')
plt.title("Class Distribution — Wine Dataset")
plt.ylabel("Count")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Preprocess & Split

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.20, random_state=42, stratify=y)
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

## Single Decision Tree Baseline

In [ ]:
base_tree = decision_tree_classifier(max_depth=4)
base_tree.fit(X_train, y_train)
base_acc = base_tree.score(X_test, y_test)
print(f"Single Decision Tree — Test Accuracy: {base_acc:.4f}")

## Hard Voting Classifier

Combines a Decision Tree, KNN, and Logistic Regression by majority vote.

In [ ]:
tree  = decision_tree_classifier(max_depth=4)
knn   = KNN(k=5)
lr    = LogisticRegression(learning_rate=0.1, n_iterations=300)

voter = hard_voting_classifier(classifiers=[tree, knn, lr])
voter.fit(X_train, y_train)
voter_acc = voter.score(X_test, y_test)
print(f"Hard Voting Classifier — Test Accuracy: {voter_acc:.4f}")

## Bagging Classifier

In [ ]:
results_bag = {}
for n_est in [5, 10, 25, 50]:
    bag = bagging_classifier(n_estimators=n_est, max_depth=4, random_state=42)
    bag.fit(X_train, y_train)
    results_bag[n_est] = bag.score(X_test, y_test)
    print(f"  Bagging n_estimators={n_est:>3}: accuracy={results_bag[n_est]:.4f}")

## Random Forest Classifier

In [ ]:
results_rf = {}
for n_est in [5, 10, 25, 50]:
    rf = random_forest_classifier(n_estimators=n_est, max_depth=4,
                                   max_features='sqrt', random_state=42)
    rf.fit(X_train, y_train)
    results_rf[n_est] = rf.score(X_test, y_test)
    print(f"  Random Forest n_estimators={n_est:>3}: accuracy={results_rf[n_est]:.4f}")

## Model Comparison

In [ ]:
best_bag = max(results_bag, key=results_bag.get)
best_rf  = max(results_rf,  key=results_rf.get)

models = ['Single Tree', 'Hard Voting',
          f'Bagging
(n={best_bag})', f'Random Forest
(n={best_rf})']
accs   = [base_acc, voter_acc, results_bag[best_bag], results_rf[best_rf]]
colors = ['#aaa', '#e9c46a', '#457b9d', '#2a9d8f']

plt.figure(figsize=(9,5))
bars = plt.bar(models, accs, color=colors, edgecolor='white', width=0.5)
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
             f'{acc:.4f}', ha='center', va='bottom', fontsize=10)
plt.ylim(0.8, 1.05)
plt.ylabel("Test Accuracy")
plt.title("Ensemble Methods vs Single Tree")
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## n_estimators vs Accuracy

In [ ]:
plt.figure(figsize=(9,5))
plt.plot(list(results_bag.keys()), list(results_bag.values()),
         'o-', color='steelblue', label='Bagging')
plt.plot(list(results_rf.keys()),  list(results_rf.values()),
         's-', color='tomato', label='Random Forest')
plt.axhline(base_acc, color='gray', linestyle='--', label='Single Tree')
plt.xlabel("n_estimators")
plt.ylabel("Test Accuracy")
plt.title("Accuracy vs Number of Trees")
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

## Key Takeaways

- **Hard Voting** leverages diverse model types to reduce individual weaknesses.
- **Bagging** reduces variance by training trees on different data samples.
- **Random Forest** further decorrelates trees with random feature subsets.
- All three methods consistently outperform a single decision tree.
